# MLFlow Search Documentation

This notebook demonstrates how to use MLFlow to track asteroid search
experiments with `find-asteroids`, and how to list experiments and
print compiled results.

**Prerequisites:** Install the pipeline extras to get MLFlow support:

```bash
pip install "find-asteroids[pipeline]"
```

The notebook will:
1. Run a `find-asteroids` search under an MLFlow experiment, logging all
   search parameters via `mlflow.log_param`.
2. List all experiments in the MLFlow tracking store.
3. List the runs recorded under the experiment, printing their parameters
   and tags.
4. Compile the results across all runs and print a summary of each table.

## Setup

In [ ]:
import tempfile
from pathlib import Path

import mlflow
from mlflow.tracking import MlflowClient

from find_asteroids.search import run_search_mlflow
from find_asteroids.results import compile_results_astropy

### Configuration

Set up data paths, the experiment name, the tracking URI, and search parameters.

Using a SQLite database (`"sqlite:///mlflow.db"`) as the tracking URI persists
experiment data between Python sessions.  Set `tracking_uri = None` to use the
default local `mlruns/` directory instead.

In [ ]:
# Data files bundled with this repository (relative to docs/).
catalog = Path("catalog.ecsv")
psfs    = Path("psfs.ecsv")

# MLFlow experiment name.  Change this to group related runs together.
experiment = "asteroid-search-example"

# MLFlow tracking URI.  Using a SQLite database persists experiment data
# between Python sessions.  Set to None to use the default local
# mlruns/ directory instead.
tracking_uri = "sqlite:///mlflow.db"

# Search parameters
velocity    = [0.1, 0.5]   # deg / day  [min, max]
angle       = [0, 359.99]  # deg        [min, max]
dx          = 10            # bin-width in PSF units (i.e. 10 × median PSF width)
num_results = 10

# Artifact location for the experiment.  Using a system temp directory so that
# artifacts are stored outside the project directory.  This directory persists
# until the OS removes it; delete it manually when no longer needed.
artifact_location = tempfile.mkdtemp()


## 1. Run a Search and Record it with MLFlow

Search parameters are passed as **keyword arguments** so that `run_search_mlflow`
automatically logs each one via `mlflow.log_param`.  The `results_dir` is created
inside a temporary directory; the results are uploaded to the tracking store as
artifacts, so the temporary directory can safely be removed after the run.

In [ ]:
run_id = run_search_mlflow(
    experiment,
    tracking_uri=tracking_uri,
    artifact_location=artifact_location,
    tags=[("dataset", "example-catalog")],
    catalog=str(catalog),
    psfs=str(psfs),
    velocity=velocity,
    angle=angle,
    dx=dx,
    num_results=num_results,
    results_dir=artifact_location,  # store run outputs inside the artifact location
)

print(f"MLFlow run ID: {run_id}")

## 2. List All Experiments

Use `MlflowClient.search_experiments()` to enumerate every experiment
in the tracking store.

In [ ]:
mlflow.set_tracking_uri(tracking_uri)
client = MlflowClient()
experiments = client.search_experiments()

print("Experiments:")
for exp in experiments:
    print(
        f"  id={exp.experiment_id!r:>4}  "
        f"name={exp.name!r}  "
        f"artifact_location={exp.artifact_location!r}"
    )

## 3. Inspect Runs in an Experiment

Use `MlflowClient.search_runs()` to list all runs under the experiment.
Each run shows its run ID, status, logged parameters, and custom tags.

In [ ]:
exp = client.get_experiment_by_name(experiment)
runs = client.search_runs(
    experiment_ids=[exp.experiment_id],
    filter_string="",
    max_results=5000,
)

print(f"Runs in experiment '{experiment}' ({len(runs)} total):")
for run in runs:
    # Omit internal mlflow.* tag keys
    user_tags = {
        k: v
        for k, v in run.data.tags.items()
        if not k.startswith("mlflow.")
    }
    print(
        f"\n  run_id : {run.info.run_id}\n"
        f"  status : {run.info.status}\n"
        f"  params : {run.data.params}\n"
        f"  tags   : {user_tags}"
    )

## 4. Compile and Print Results

Use `compile_results_astropy` with `reader="mlflow"` to download artifacts
from all runs in the experiment and stack them into four Astropy tables:

| Table | Contents |
|-------|----------|
| `result` | Hough-space peak: x, y, direction, vote count *n* |
| `tracklet` | Refined on-sky trajectory: velocities, positions, uncertainties |
| `points` | Catalog detections that voted for each result |
| `gathered` | Original catalog entries matched to each result |

> **Note:** The tracking URI must be set via `mlflow.set_tracking_uri()` before
> calling `compile_results_astropy` so that the underlying `read_results_mlflow`
> uses the correct tracking store.

In [ ]:
# Set the tracking URI before compiling so the correct store is used.
mlflow.set_tracking_uri(tracking_uri)

for name, table in compile_results_astropy(
    experiment, reader="mlflow", output_format="ecsv"
):
    print(f"\n-- Table: '{name}' --")
    print(f"   rows    : {len(table)}")
    print(f"   columns : {table.colnames}")
    table.pprint(max_lines=5, max_width=200)